# 11 — Emergency Penalty (RQ7)

**Research Question:** Is emergency presentation a signal of upstream system failure?

When the same procedure is performed as emergency instead of elective, the outcome penalty
(longer LOS, higher mortality, higher cost) quantifies a preventable system failure.

**Report:** `reports/11_emergency_penalty.md`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from shared import (
    load_kidney, load_hospital_tags, load_cnes_enriched,
    setup_plot_style, save_plot, save_metrics,
    CATEGORY_MAP, PROC_NAMES, CITY_NAMES,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
cnes = load_cnes_enriched()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")

emerg = kidney[kidney["is_emergency"] == 1]
elect = kidney[kidney["is_emergency"] == 0]
print(f"Emergency: {len(emerg):,} ({len(emerg)/len(kidney)*100:.1f}%)")
print(f"Elective:  {len(elect):,} ({len(elect)/len(kidney)*100:.1f}%)")

Loaded 206,500 admissions, 510 hospitals
Emergency: 116,672 (56.5%)
Elective:  89,828 (43.5%)


## 1. Same-Procedure Emergency vs Elective Delta

For each procedure code with sufficient volume in both admission types, compare LOS, mortality, and cost.
This is the core test — controlling for procedure eliminates procedure-mix confounding.

In [2]:
MIN_PER_GROUP = 50

proc_adm = kidney.groupby(["PROC_REA", "is_emergency"]).agg(
    n=pd.NamedAgg(column="DIAS_PERM", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    median_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="median"),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
).reset_index()

# Keep procedures with >= MIN_PER_GROUP in BOTH emergency and elective
both_groups = proc_adm.groupby("PROC_REA").filter(
    lambda g: (g["is_emergency"] == 0).any() and (g["is_emergency"] == 1).any()
    and g[g["is_emergency"] == 0]["n"].values[0] >= MIN_PER_GROUP
    and g[g["is_emergency"] == 1]["n"].values[0] >= MIN_PER_GROUP
)

procs_to_compare = both_groups["PROC_REA"].unique()
print(f"Procedures with >={MIN_PER_GROUP} in both groups: {len(procs_to_compare)}")

# Compute deltas
deltas = []
for proc in procs_to_compare:
    e = both_groups[(both_groups["PROC_REA"] == proc) & (both_groups["is_emergency"] == 1)].iloc[0]
    el = both_groups[(both_groups["PROC_REA"] == proc) & (both_groups["is_emergency"] == 0)].iloc[0]
    
    # Mann-Whitney U for LOS
    emerg_los = kidney[(kidney["PROC_REA"] == proc) & (kidney["is_emergency"] == 1)]["DIAS_PERM"]
    elect_los = kidney[(kidney["PROC_REA"] == proc) & (kidney["is_emergency"] == 0)]["DIAS_PERM"]
    u_stat, u_p = stats.mannwhitneyu(emerg_los, elect_los, alternative="two-sided")
    
    deltas.append({
        "proc": proc,
        "proc_name": PROC_NAMES.get(proc, proc),
        "category": CATEGORY_MAP.get(proc, "OTHER"),
        "n_emerg": int(e["n"]),
        "n_elect": int(el["n"]),
        "los_emerg": e["mean_los"],
        "los_elect": el["mean_los"],
        "los_delta": e["mean_los"] - el["mean_los"],
        "los_pct_penalty": (e["mean_los"] / max(el["mean_los"], 0.01) - 1) * 100,
        "mort_emerg": e["mortality"] * 100,
        "mort_elect": el["mortality"] * 100,
        "mort_delta_pct": (e["mortality"] - el["mortality"]) * 100,
        "cost_emerg": e["mean_cost"],
        "cost_elect": el["mean_cost"],
        "cost_penalty_pct": (e["mean_cost"] / max(el["mean_cost"], 0.01) - 1) * 100,
        "los_mwu_p": u_p,
    })

deltas_df = pd.DataFrame(deltas).sort_values("los_delta", ascending=False)

print("\n=== EMERGENCY PENALTY BY PROCEDURE ===")
print(deltas_df[["proc_name", "category", "n_emerg", "n_elect",
                  "los_emerg", "los_elect", "los_delta", "los_pct_penalty",
                  "mort_emerg", "mort_elect", "los_mwu_p"]].to_string(index=False))

Procedures with >=50 in both groups: 20



=== EMERGENCY PENALTY BY PROCEDURE ===
                     proc_name       category  n_emerg  n_elect  los_emerg  los_elect  los_delta  los_pct_penalty  mort_emerg  mort_elect     los_mwu_p
                    0409010200          OTHER       99      192   9.171717   4.619792   4.551926        98.530969    4.040404    1.041667  6.506105e-10
                    0409010286          OTHER      102       59  10.009804   5.983051   4.026753        67.302672    8.823529    5.084746  7.493932e-04
                    0409010219          OTHER       70      207   7.714286   4.405797   3.308489        75.093985    5.714286    0.000000  1.758400e-09
              ESWL Lithotripsy       SURGICAL      513     2271   3.169591   0.385733   2.783857       721.705520    0.194932    0.000000 1.940252e-238
            Kidney Exploration       SURGICAL      320      234   7.450000   5.354701   2.095299        39.130088    5.312500    0.427350  1.375441e-10
                   Nephrectomy       SURGICAL   

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(20, max(5, len(deltas_df) * 0.5)))

# LOS penalty
labels = deltas_df["proc_name"].tolist()
y = range(len(labels))

colors = ["#DC2626" if d > 0 else "#059669" for d in deltas_df["los_delta"]]
axes[0].barh(y, deltas_df["los_delta"], color=colors)
axes[0].set_yticks(y)
axes[0].set_yticklabels(labels, fontsize=8)
axes[0].set_title("LOS Penalty (Emergency − Elective)")
axes[0].set_xlabel("Additional days")
axes[0].axvline(0, color="black", linewidth=0.5)
axes[0].invert_yaxis()
for i, (d, p) in enumerate(zip(deltas_df["los_delta"], deltas_df["los_mwu_p"])):
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    axes[0].text(d + 0.05 if d >= 0 else d - 0.05, i, f"{d:+.2f}d {sig}",
                 va="center", ha="left" if d >= 0 else "right", fontsize=7)

# Mortality penalty
axes[1].barh(y, deltas_df["mort_delta_pct"], color=colors)
axes[1].set_yticks(y)
axes[1].set_yticklabels(labels, fontsize=8)
axes[1].set_title("Mortality Penalty (Emergency − Elective)")
axes[1].set_xlabel("Additional mortality (pp)")
axes[1].axvline(0, color="black", linewidth=0.5)
axes[1].invert_yaxis()

# Cost penalty %
axes[2].barh(y, deltas_df["cost_penalty_pct"], color=colors)
axes[2].set_yticks(y)
axes[2].set_yticklabels(labels, fontsize=8)
axes[2].set_title("Cost Penalty (Emergency vs Elective)")
axes[2].set_xlabel("% additional cost")
axes[2].axvline(0, color="black", linewidth=0.5)
axes[2].invert_yaxis()

fig.suptitle("Emergency Penalty — Same Procedure, Different Pathway", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "emergency_penalty_by_proc", prefix="11")

  Saved: 11_emergency_penalty_by_proc.png


## 2. Age-Stratified Emergency Penalty

Does the emergency penalty hold after controlling for age? Older patients may both
present more as emergency AND have longer stays — a classic confounder.

In [4]:
age_bins = [0, 40, 60, 80, 100]
age_labels = ["<40", "40-59", "60-79", "80+"]
kidney["age_group"] = pd.cut(kidney["age"], bins=age_bins, labels=age_labels, right=False)

age_strat = kidney.groupby(["age_group", "is_emergency"], observed=False).agg(
    n=pd.NamedAgg(column="DIAS_PERM", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    mean_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for label in age_labels:
    grp = age_strat[age_strat["age_group"] == label]
    e_row = grp[grp["is_emergency"] == 1]
    el_row = grp[grp["is_emergency"] == 0]
    if e_row.empty or el_row.empty:
        continue

x = np.arange(len(age_labels))
width = 0.35
emerg_los = [age_strat[(age_strat["age_group"]==a) & (age_strat["is_emergency"]==1)]["mean_los"].values[0]
             for a in age_labels]
elect_los = [age_strat[(age_strat["age_group"]==a) & (age_strat["is_emergency"]==0)]["mean_los"].values[0]
             for a in age_labels]
axes[0].bar(x - width/2, emerg_los, width, label="Emergency", color="#DC2626")
axes[0].bar(x + width/2, elect_los, width, label="Elective", color="#2563EB")
axes[0].set_xticks(x)
axes[0].set_xticklabels(age_labels)
axes[0].set_title("Mean LOS by Age Group")
axes[0].set_ylabel("Days")
axes[0].legend()

emerg_mort = [age_strat[(age_strat["age_group"]==a) & (age_strat["is_emergency"]==1)]["mortality"].values[0]*100
              for a in age_labels]
elect_mort = [age_strat[(age_strat["age_group"]==a) & (age_strat["is_emergency"]==0)]["mortality"].values[0]*100
              for a in age_labels]
axes[1].bar(x - width/2, emerg_mort, width, label="Emergency", color="#DC2626")
axes[1].bar(x + width/2, elect_mort, width, label="Elective", color="#2563EB")
axes[1].set_xticks(x)
axes[1].set_xticklabels(age_labels)
axes[1].set_title("Mortality by Age Group")
axes[1].set_ylabel("Mortality (%)")
axes[1].legend()

# Emergency rate by age group
emerg_rate = kidney.groupby("age_group", observed=False)["is_emergency"].mean() * 100
axes[2].bar(age_labels, emerg_rate.values, color="#D97706")
axes[2].set_title("Emergency Rate by Age Group")
axes[2].set_ylabel("% emergency")

fig.suptitle("Age-Stratified Emergency Penalty", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "age_stratified_penalty", prefix="11")

print("Emergency penalty persists within age groups:")
for a in age_labels:
    e_los = age_strat[(age_strat["age_group"]==a) & (age_strat["is_emergency"]==1)]["mean_los"].values[0]
    el_los = age_strat[(age_strat["age_group"]==a) & (age_strat["is_emergency"]==0)]["mean_los"].values[0]
    print(f"  {a}: Emergency {e_los:.2f}d vs Elective {el_los:.2f}d (delta: {e_los-el_los:+.2f}d)")

  Saved: 11_age_stratified_penalty.png
Emergency penalty persists within age groups:
  <40: Emergency 2.69d vs Elective 1.80d (delta: +0.89d)
  40-59: Emergency 2.92d vs Elective 1.73d (delta: +1.19d)
  60-79: Emergency 3.68d vs Elective 1.81d (delta: +1.86d)
  80+: Emergency 5.16d vs Elective 2.19d (delta: +2.97d)


## 3. Geographic Access & Emergency Rate

Do cities without surgical capability or with high migration rates show higher emergency rates?
This tests the "system failure" hypothesis — patients present as emergency because they couldn't
access scheduled care.

In [5]:
# City-level emergency rates
city_stats = kidney.groupby("city_name").agg(
    n=pd.NamedAgg(column="city_name", aggfunc="count"),
    emergency_rate=pd.NamedAgg(column="is_emergency", aggfunc="mean"),
    has_surgery=pd.NamedAgg(column="proc_category",
                            aggfunc=lambda x: (x == "SURGICAL").any()),
    migration_inflow=pd.NamedAgg(column="migrated", aggfunc="mean"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).reset_index()
city_stats = city_stats[city_stats["n"] >= 30]

# Compare emergency rate: cities with vs without surgery
has_surg = city_stats[city_stats["has_surgery"]]
no_surg = city_stats[~city_stats["has_surgery"]]

print(f"Cities with surgical capability: {len(has_surg)}")
print(f"  Mean emergency rate: {has_surg['emergency_rate'].mean()*100:.1f}%")
print(f"Cities without surgical capability: {len(no_surg)}")
print(f"  Mean emergency rate: {no_surg['emergency_rate'].mean()*100:.1f}%")

if len(no_surg) >= 5:
    u_stat, u_p = stats.mannwhitneyu(no_surg["emergency_rate"], has_surg["emergency_rate"],
                                      alternative="greater")
    print(f"  Mann-Whitney U p = {u_p:.4f} (no surgery > has surgery)")

# Migrated vs non-migrated emergency rates
migrated_emerg = kidney[kidney["migrated"]]["is_emergency"].mean() * 100
local_emerg = kidney[~kidney["migrated"]]["is_emergency"].mean() * 100
print(f"\nMigrated patients emergency rate: {migrated_emerg:.1f}%")
print(f"Local patients emergency rate: {local_emerg:.1f}%")
u2, p2 = stats.mannwhitneyu(
    kidney[kidney["migrated"]]["is_emergency"],
    kidney[~kidney["migrated"]]["is_emergency"],
    alternative="two-sided"
)
print(f"Mann-Whitney U p = {p2:.2e}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Surgical capability vs emergency rate
labels_surg = ["Has Surgery", "No Surgery"]
rates_surg = [has_surg["emergency_rate"].mean()*100, no_surg["emergency_rate"].mean()*100 if len(no_surg) > 0 else 0]
axes[0].bar(labels_surg, rates_surg, color=["#059669", "#DC2626"])
axes[0].set_title("Emergency Rate by Surgical Capability")
axes[0].set_ylabel("Emergency rate (%)")
for i, v in enumerate(rates_surg):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")

# Migration vs emergency
labels_mig = ["Local", "Migrated"]
rates_mig = [local_emerg, migrated_emerg]
axes[1].bar(labels_mig, rates_mig, color=["#2563EB", "#D97706"])
axes[1].set_title("Emergency Rate: Local vs Migrated Patients")
axes[1].set_ylabel("Emergency rate (%)")
for i, v in enumerate(rates_mig):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")

# Scatter: city migration rate vs emergency rate
axes[2].scatter(city_stats["migration_inflow"]*100, city_stats["emergency_rate"]*100,
               s=city_stats["n"]/10, alpha=0.5, color="#7C3AED")
r, p_corr = stats.pearsonr(city_stats["migration_inflow"], city_stats["emergency_rate"])
axes[2].set_title(f"City Migration vs Emergency Rate (r={r:.2f}, p={p_corr:.3f})")
axes[2].set_xlabel("Migration inflow rate (%)")
axes[2].set_ylabel("Emergency rate (%)")

fig.suptitle("Geographic Access & Emergency Presentation", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "geographic_emergency", prefix="11")

Cities with surgical capability: 128
  Mean emergency rate: 64.7%
Cities without surgical capability: 54
  Mean emergency rate: 96.6%
  Mann-Whitney U p = 0.0000 (no surgery > has surgery)

Migrated patients emergency rate: 48.9%
Local patients emergency rate: 60.9%
Mann-Whitney U p = 0.00e+00


  Saved: 11_geographic_emergency.png


## 4. Emergency Trend — Is It Improving?

The emergency rate has been declining (58% → 49%). Is this improvement uniform across procedures?

In [6]:
k16 = kidney[kidney["year"] >= 2016]
top_procs = k16["proc_name"].value_counts().head(6).index

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, proc in zip(axes.flat, top_procs):
    proc_data = k16[k16["proc_name"] == proc]
    trend = proc_data.groupby("year")["is_emergency"].mean() * 100
    ax.plot(trend.index, trend.values, "o-", color="#DC2626")
    ax.set_title(proc, fontsize=9)
    ax.set_ylabel("% Emergency")
    ax.set_xlabel("Year")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Emergency Rate Trend by Procedure", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "emergency_trend_by_proc", prefix="11")

  Saved: 11_emergency_trend_by_proc.png


## 5. Counterfactual: Savings from Avoiding Emergency Pathway

If emergencies that could have been elective had the elective LOS and mortality,
how many bed-days and deaths would be saved?

This is a **crude upper-bound estimate**, not a causal claim.

In [7]:
# For each procedure with both pathways, estimate savings
total_excess_bed_days = 0
total_excess_deaths = 0
total_excess_cost = 0
counterfactual_details = []

for _, row in deltas_df.iterrows():
    if row["los_delta"] <= 0:
        continue
    excess_days = row["los_delta"] * row["n_emerg"]
    excess_deaths = row["mort_delta_pct"] / 100 * row["n_emerg"]
    excess_cost = (row["cost_emerg"] - row["cost_elect"]) * row["n_emerg"]
    
    total_excess_bed_days += excess_days
    total_excess_deaths += max(0, excess_deaths)
    total_excess_cost += max(0, excess_cost)
    
    counterfactual_details.append({
        "proc_name": row["proc_name"],
        "n_emerg": row["n_emerg"],
        "excess_bed_days": int(excess_days),
        "excess_deaths": round(excess_deaths, 1),
        "excess_cost_brl": int(excess_cost),
    })

cf_df = pd.DataFrame(counterfactual_details).sort_values("excess_bed_days", ascending=False)

print("=== COUNTERFACTUAL: IF EMERGENCIES HAD ELECTIVE OUTCOMES ===")
print(f"Total excess bed-days: {total_excess_bed_days:,.0f}")
print(f"Total excess deaths: {total_excess_deaths:,.0f}")
print(f"Total excess cost: R${total_excess_cost:,.0f}")
print(f"\nBreakdown by procedure:")
print(cf_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, max(4, len(cf_df) * 0.5)))
ax.barh(range(len(cf_df)), cf_df["excess_bed_days"], color="#DC2626")
ax.set_yticks(range(len(cf_df)))
ax.set_yticklabels(cf_df["proc_name"], fontsize=9)
ax.set_title("Excess Bed-Days from Emergency Pathway (per procedure)")
ax.set_xlabel("Excess bed-days")
ax.invert_yaxis()
for i, v in enumerate(cf_df["excess_bed_days"]):
    ax.text(v + 100, i, f"{v:,}", va="center", fontsize=8)
fig.tight_layout()
save_plot(fig, "counterfactual_savings", prefix="11")

=== COUNTERFACTUAL: IF EMERGENCIES HAD ELECTIVE OUTCOMES ===
Total excess bed-days: 114,811
Total excess deaths: 375
Total excess cost: R$10,732,267

Breakdown by procedure:
                     proc_name  n_emerg  excess_bed_days  excess_deaths  excess_cost_brl
         Open Ureterolithotomy    29146            45579          195.3          6508013
         Ureteroscopy (modern)    11774            16377           25.0          -998346
Diagnostic Imaging (Urography)    38289            14753           22.5          2674350
           Clinical Management     8426            13401           47.2           327299
           Surgical Management    10303             8534           23.0          -675136
             Ureteral Catheter     6670             7309           16.8            51661
              ESWL Lithotripsy      513             1428            1.0           -15539
                Pyelolithotomy      974             1084            0.9          -138579
                      JJ 

## 6. Sub-diagnosis Analysis

Do different stone locations (N200 kidney, N201 ureter, N202 both) show different emergency patterns?

In [8]:
subdiag_emerg = kidney.groupby("DIAG_PRINC").agg(
    n=pd.NamedAgg(column="DIAG_PRINC", aggfunc="count"),
    emergency_rate=pd.NamedAgg(column="is_emergency", aggfunc="mean"),
    mean_los_emerg=pd.NamedAgg(column="DIAS_PERM",
        aggfunc=lambda x: x[kidney.loc[x.index, "is_emergency"]==1].mean()),
    mean_los_elect=pd.NamedAgg(column="DIAS_PERM",
        aggfunc=lambda x: x[kidney.loc[x.index, "is_emergency"]==0].mean()),
    mortality=pd.NamedAgg(column="MORTE", aggfunc="mean"),
).reset_index()
subdiag_emerg["los_penalty"] = subdiag_emerg["mean_los_emerg"] - subdiag_emerg["mean_los_elect"]

print("=== SUB-DIAGNOSIS EMERGENCY PATTERNS ===")
print(subdiag_emerg.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

subdiag_emerg.set_index("DIAG_PRINC")[["emergency_rate"]].mul(100).plot.bar(
    ax=axes[0], color="#DC2626", legend=False)
axes[0].set_title("Emergency Rate by Sub-diagnosis")
axes[0].set_ylabel("% Emergency")
axes[0].tick_params(axis="x", rotation=0)

subdiag_emerg.set_index("DIAG_PRINC")[["los_penalty"]].plot.bar(
    ax=axes[1], color="#D97706", legend=False)
axes[1].set_title("LOS Penalty (Emergency − Elective) by Sub-diagnosis")
axes[1].set_ylabel("Additional days")
axes[1].tick_params(axis="x", rotation=0)

fig.suptitle("Sub-Diagnosis Emergency Analysis", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "subdiagnosis_emergency", prefix="11")

=== SUB-DIAGNOSIS EMERGENCY PATTERNS ===
DIAG_PRINC     n  emergency_rate  mean_los_emerg  mean_los_elect  mortality  los_penalty
       N20  1918        0.653806        2.947368        1.415663   0.004171     1.531706
      N200 72093        0.519454        3.086785        2.021735   0.003648     1.065049
      N201 80523        0.619115        2.828977        1.515487   0.002546     1.313490
      N202 39539        0.472344        3.137235        1.773283   0.003566     1.363952
      N209 12427        0.759636        3.082627        1.670907   0.007806     1.411720


  Saved: 11_subdiagnosis_emergency.png


## Summary Metrics

In [9]:
# Weighted average emergency penalty across procedures with both pathways
weighted_los_penalty = (deltas_df["los_delta"] * deltas_df["n_emerg"]).sum() / deltas_df["n_emerg"].sum()
procs_with_sig_penalty = (deltas_df["los_mwu_p"] < 0.05).sum()

metrics = {
    "procedures_compared": int(len(deltas_df)),
    "procedures_significant_penalty": int(procs_with_sig_penalty),
    "weighted_avg_los_penalty_days": round(float(weighted_los_penalty), 2),
    "overall_emergency_rate_pct": round(float(kidney["is_emergency"].mean() * 100), 1),
    "emergency_mean_los": round(float(emerg["DIAS_PERM"].mean()), 2),
    "elective_mean_los": round(float(elect["DIAS_PERM"].mean()), 2),
    "emergency_mortality_pct": round(float(emerg["MORTE"].mean() * 100), 3),
    "elective_mortality_pct": round(float(elect["MORTE"].mean() * 100), 3),
    "migrated_emergency_rate_pct": round(float(migrated_emerg), 1),
    "local_emergency_rate_pct": round(float(local_emerg), 1),
    "counterfactual_excess_bed_days": int(total_excess_bed_days),
    "counterfactual_excess_deaths": int(total_excess_deaths),
    "counterfactual_excess_cost_brl": int(total_excess_cost),
}
save_metrics(metrics, "11_emergency_penalty")

print("\n=== EMERGENCY PENALTY METRICS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 11_emergency_penalty.json

=== EMERGENCY PENALTY METRICS ===
  procedures_compared: 20
  procedures_significant_penalty: 20
  weighted_avg_los_penalty_days: 1.0
  overall_emergency_rate_pct: 56.5
  emergency_mean_los: 2.98
  elective_mean_los: 1.78
  emergency_mortality_pct: 0.513
  elective_mortality_pct: 0.129
  migrated_emergency_rate_pct: 48.9
  local_emergency_rate_pct: 60.9
  counterfactual_excess_bed_days: 114811
  counterfactual_excess_deaths: 374
  counterfactual_excess_cost_brl: 10732267
